In [ ]:
import os
at_colab = 'COLAB_GPU' in os.environ

if at_colab:
    %pip install folium
    %pip install mapclassify
    %pip install selenium webdriver-manager
    %pip install wget

In [ ]:
# Standard library imports
import os
import zipfile
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from scipy.spatial import KDTree
import matplotlib.pyplot as plt

CKAN (Comprehensive Knowledge Archive Network) is an open-source data management system that is widely used for publishing, sharing, and accessing datasets. It provides a structured way to organize and distribute data, often in the context of open data portals.

The **Humanitarian Data Exchange (HDX)**, which hosts the dataset you are referring to, is built on CKAN. CKAN provides an API that allows users to programmatically access dataset metadata, including download links for resources.

In this context, CKAN allows you to:
- Query dataset metadata.
- Retrieve file download links programmatically.
- Automate the downloading of multiple resources from a dataset.

Would you like help with modifying the script to fit your specific needs?

In [ ]:
def fetch_dataset_metadata(dataset_id: str) -> dict:
    """
    Fetches the metadata of a CKAN dataset using the CKAN API.

    Args:
        dataset_id (str): The unique identifier of the dataset.

    Returns:
        dict: The dataset's metadata.
    """
    api_url = f"https://data.humdata.org/api/3/action/package_show?id={dataset_id}"
    response = requests.get(api_url)
    response.raise_for_status()  # Raises an HTTPError for bad responses
    return response.json()['result']

def download_file(url: str, destination: Path) -> None:
    """
    Downloads a file from a given URL to a specified local destination.

    Args:
        url (str): The URL of the file to download.
        destination (Path): The local file path where the file will be saved.
    """
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with destination.open('wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)

def download_all_resources(dataset_id: str, download_folder: Path) -> None:
    """
    Downloads all resources associated with a CKAN dataset into a specified folder.

    Args:
        dataset_id (str): The unique identifier of the dataset.
        download_folder (Path): The local folder where the resources will be saved.
    """
    # Ensure the folder exists
    download_folder.mkdir(parents=True, exist_ok=True)

    metadata = fetch_dataset_metadata(dataset_id)
    resources = metadata.get('resources', [])

    for resource in resources:
        resource_url = resource['url']
        resource_name = resource['name']
        destination_path = download_folder / resource_name

        print(f"Downloading {resource_name} from {resource_url} to {destination_path}...")
        download_file(resource_url, destination_path)
        print(f"Downloaded {resource_name} successfully.")

In [ ]:
# Example usage
dataset_id = 'netherlands-high-resolution-population-density-maps-demographic-estimates'
download_folder = Path("datasets/netherlands_population")  # Define the target folder

In [ ]:
download_all_resources(dataset_id, download_folder)

In [ ]:
# Define the folder paths
zip_folder = download_folder
extract_folder = 'extracted_data'

# Ensure extraction folder exists
Path(extract_folder).mkdir(parents=True, exist_ok=True)

# Step 1: Extract all ZIP files
csv_files = []
for zip_path in Path(zip_folder).glob('*.zip'):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
        csv_files.extend([str(Path(extract_folder) / f) for f in zip_ref.namelist() if f.endswith('.csv')])

# Step 2: Read and combine CSV files
dfs = {}
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        # Identify coordinate columns (assuming common names)
        possible_lat = [col for col in df.columns if 'lat' in col.lower()]
        possible_lon = [col for col in df.columns if 'lon' in col.lower() or 'long' in col.lower()]

        if possible_lat and possible_lon:
            df = df.rename(columns={possible_lat[0]: 'latitude', possible_lon[0]: 'longitude'})
            df = df.set_index(['latitude', 'longitude'])

        # Store DataFrame with a unique name
        dataset_name = Path(csv_file).stem  # Use filename as dataset key
        dfs[dataset_name] = df
    except Exception as e:
        print(f"Error reading {csv_file}: {e}")

In [ ]:
[Path(f).stem for f in csv_files]

In [ ]:
[f'{df.shape[0]:,}' for df in dfs.values()]

In [ ]:
largest_df_key = max(dfs, key=lambda k: dfs[k].shape[0])
largest_df_key

In [ ]:
largest_df = dfs[largest_df_key]

lat_lon_list = largest_df.index.values.tolist()

# Build KDTree
tree = KDTree(lat_lon_list)

# Query the nearest neighbor for each point (excluding itself)
# k=2 includes the point itself and its nearest neighbor
distances, indices = tree.query(lat_lon_list, k=2)

# first distance is to itself, second to the next closest
assert distances[:,0].min() == 0
assert distances[:,0].max() == 0

# Get the minimum nonzero distance
min_distance = distances[:, 1] # Exclude the self-match at index 0

In [ ]:
# Create the figure
fig, ax = plt.subplots(figsize=(15, 1))

# Create a horizontal box plot with improved visibility
box = ax.boxplot(
    min_distance,
    vert=False,
    patch_artist=True,  # Enables color fill
    showfliers=True,  # Show outliers
    widths=0.4,  # Make the box wider
    boxprops={'facecolor': 'lightblue', 'edgecolor': 'black', 'linewidth': 2.5},
    medianprops={'color': 'red', 'linewidth': 3},
    whiskerprops={'color': 'black', 'linewidth': 2},
    capprops={'color': 'black', 'linewidth': 2},
    flierprops={'marker': 'o', 'markersize': 6, 'markerfacecolor': 'red', 'alpha': 0.6}
)

# Remove y-axis ticks
ax.set_yticks([])

# # Labels and title
# ax.set_xlabel('Distance')
# ax.set_title('Enhanced Horizontal Box Plot')
plt.savefig('box_plot_grid.png', format="png", bbox_inches="tight", dpi=300)

# Show the plot
plt.show()


In [ ]:
if not at_colab:
    import pyperclip
    pyperclip.copy(
        pd.Series(min_distance).describe().to_frame().T.to_latex(
            index=False,
            escape=True
        )
    )

In [ ]:
lat_lon = lat_lon_list[min_distance.argmax()]
print(f'{lat_lon} {largest_df.loc[lat_lon].Population}')

In [ ]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="geo_explorer")
location = geolocator.reverse(lat_lon, exactly_one=True)

print(location.address)

In [ ]:
import os
import wget

url = 'https://raw.githubusercontent.com/gromicho/tools/main/jg_folium.py'
filename = 'jg_folium.py'

# Ensure the file is removed before downloading
if os.path.exists(filename):
    os.remove(filename)

# Download the file (this will now always download the latest version)
wget.download(url, filename)

import jg_folium as jg

In [ ]:
import folium
m = folium.Map(location=lat_lon, zoom_start=15)
folium.Marker(lat_lon).add_to(m)
jg.FoliumToPng( m, 'remote_pop' )
m

In [ ]:
from collections import defaultdict

group = defaultdict(list)
for name,df in dfs.items():
    group[frozenset(df.index.values)].append(name)

In [ ]:
group.values()

In [ ]:
firsts = [values[0] for values in group.values()]
tree = KDTree(dfs[firsts[0]].index.values.tolist())
max_dist = max(
    max(tree.query(dfs[name].index.values.tolist())[0])
    for name in firsts[1:]
)
max_dist


In [ ]:
merged = {
    key : pd.concat(
        [dfs[name] for name in values],
        axis=1,
        join='outer',
        keys=values
    )
    for key,values in group.items()
}

In [ ]:
keys = list(merged.keys())
final = merged[keys[0]]

B = final.index.values.tolist()
treeB = KDTree(B)

for key in keys[1:]:
    df = merged[key]
    A = df.index.values.tolist()

    distances, indices = treeB.query(A)
    print(f'{min(distances)} {max(distances)}')

    mapping = {A[i]:B[j] for i,j in enumerate(indices)}

    df.index = pd.MultiIndex.from_tuples(
        [mapping.get(idx, idx) for idx in df.index],
        names=df.index.names
    )

    final = pd.concat([final, df], axis=1, join='outer').dropna()

final.columns = final.columns.get_level_values(0)

In [ ]:
final.isna().sum()

In [ ]:
dfA = merged[keys[0]]
dfB = merged[keys[1]]
A = dfA.index.values.tolist()
B = dfB.index.values.tolist()
distances, indices = KDTree(A).query(B)

In [ ]:
missing=sorted(set(range(max(indices)))-set(indices))

In [ ]:
dfA.iloc[missing,0].sum()

In [ ]:
(dfA == 0).sum().iloc[0], len(missing)

In [ ]:
population = final['population_nld_2019-07-01']
genders = final['NLD_men_2019-08-03'] + final['NLD_women_2019-08-03']

total_pop = population.sum()
total_genders = genders.sum()

f"{( total_pop - total_genders ) / total_pop * 100:.2}%"

In [ ]:
(population-genders).hist(bins=100,density=True,figsize=(10,5))
plt.savefig('gender_hist.png', format="png", bbox_inches="tight", dpi=300)

In [ ]:
from titlecase import titlecase

In [ ]:
epsg = 'EPSG:4326' # https://en.wikipedia.org/wiki/World_Geodetic_System#WGS_84

pop = final.reset_index(names=final.index.names)

rename_cols = { c : titlecase(' '.join(c.split('_')[1:-1])) 
    for c in pop.columns if c.startswith('NLD_') 
} | { c : titlecase(c.split('_')[0]) 
    for c in pop.columns if c.lower().startswith('population') 
}

In [ ]:
rename_cols

In [ ]:
pop = gpd.GeoDataFrame(
    pop, geometry=gpd.points_from_xy(pop['longitude'], pop['latitude']), crs=epsg
).rename( columns=rename_cols )

In [ ]:
pop.to_parquet('HDX_NL.parquet')

In [ ]:
pop